In [0]:
from pyspark.sql import functions as F

SOURCE_PATH = "/Volumes/ai_analytics/source/v1_bronze/messages/"
TARGET_TABLE = "ai_analytics.bronze.messages"

messages_df = (
    spark.read
    .option("basePath", SOURCE_PATH)
    .json(SOURCE_PATH)
)

messages_df.printSchema()

print("Total physical Bronze rows:", messages_df.count())

display(messages_df.limit(20))

In [0]:
assert not spark.catalog.tableExists(TARGET_TABLE), (
    f"{TARGET_TABLE} already exists. Stop before overwriting anything."
)

(
    messages_df.writeTo(TARGET_TABLE)
    .using("delta")
    .create()
)

print(f"Created {TARGET_TABLE}")

In [0]:
from pyspark.sql import functions as F

TARGET_TABLE = "ai_analytics.bronze.messages"

delta_df = spark.table(TARGET_TABLE)

print("Delta total rows:", delta_df.count())
print("Delta columns:", delta_df.columns)

delta_df.printSchema()

In [0]:
aug22_count = (
    delta_df
    .filter(F.to_date(F.col("dt")) == F.lit("2026-08-22").cast("date"))
    .count()
)

print("Delta Aug 22 rows:", aug22_count)

assert aug22_count == 29573

In [0]:
source_count = messages_df.count()
delta_count = delta_df.count()

print("V1 S3 Bronze:", source_count)
print("Delta Bronze:", delta_count)

assert delta_count == source_count

print("PASS: historical physical row count preserved")

In [0]:
display(
    spark.sql("""
        DESCRIBE HISTORY ai_analytics.bronze.messages
    """)
)

In [0]:
%sql
ALTER TABLE ai_analytics.bronze.messages
SET TBLPROPERTIES (
  'delta.appendOnly' = 'true'
);

In [0]:
spark.sql("""
ALTER TABLE ai_analytics.bronze.messages
SET TBLPROPERTIES (
  'delta.appendOnly' = 'true'
)
""")

print("Bronze messages set to append-only")

In [0]:
display(
    spark.sql("""
        SHOW TBLPROPERTIES ai_analytics.bronze.messages
    """)
)

In [0]:
from pyspark.sql import Row

SOURCE_ROOT = "/Volumes/ai_analytics/source/v1_bronze"

ENTITIES = [
    "models",
    "devices",
    "subscription_plans",
    "users",
    "user_updates",
    "conversations",
    # "messages" already migrated
    "completions",
    "model_inferences",
    "feedback",
    "errors",
    "subscriptions",
    "purchases",
    "payments",
]

results = []

for entity in ENTITIES:
    path = f"{SOURCE_ROOT}/{entity}/"

    print(f"\nReading {entity}...")

    df = (
        spark.read
        .option("basePath", path)
        .json(path)
    )

    row_count = df.count()

    results.append(
        Row(
            entity=entity,
            source_row_count=row_count,
            column_count=len(df.columns)
        )
    )

    print(
        f"{entity}: "
        f"{row_count:,} rows, "
        f"{len(df.columns)} columns"
    )

result_df = spark.createDataFrame(results)

display(result_df.orderBy("entity"))

In [0]:
EXPECTED_COUNTS = {
    "models": 8,
    "devices": 12,
    "subscription_plans": 5,
    "users": 10000,
    "user_updates": 1828,
    "conversations": 121474,
    "completions": 1175788,
    "model_inferences": 1175519,
    "feedback": 73216,
    "errors": 26097,
    "subscriptions": 13153,
    "purchases": 12321,
    "payments": 13296,
}

SOURCE_ROOT = "/Volumes/ai_analytics/source/v1_bronze"
TARGET_SCHEMA = "ai_analytics.bronze"

migration_results = []

for entity, expected_count in EXPECTED_COUNTS.items():

    source_path = f"{SOURCE_ROOT}/{entity}/"
    target_table = f"{TARGET_SCHEMA}.{entity}"

    print(f"\n--- {entity} ---")

    source_df = (
        spark.read
        .option("basePath", source_path)
        .json(source_path)
    )

    source_count = source_df.count()

    # First protect against unexpected source changes
    assert source_count == expected_count, (
        f"{entity}: expected source count {expected_count:,}, "
        f"but found {source_count:,}"
    )

    if spark.catalog.tableExists(target_table):
        print(f"{target_table} already exists — NOT overwriting")

    else:
        (
            source_df.writeTo(target_table)
            .using("delta")
            .create()
        )

        print(f"Created {target_table}")

    # Validate target
    delta_count = spark.table(target_table).count()

    assert delta_count == source_count, (
        f"{entity}: source={source_count:,}, "
        f"delta={delta_count:,}"
    )

    # Make Bronze append-only
    spark.sql(f"""
        ALTER TABLE {target_table}
        SET TBLPROPERTIES (
            'delta.appendOnly' = 'true'
        )
    """)

    migration_results.append(
        (
            entity,
            source_count,
            delta_count,
            source_count == delta_count
        )
    )

    print(
        f"PASS {entity}: "
        f"{source_count:,} source → {delta_count:,} Delta"
    )

In [0]:
result_df = spark.createDataFrame(
    migration_results,
    [
        "entity",
        "source_row_count",
        "delta_row_count",
        "counts_match"
    ]
)

display(result_df.orderBy("entity"))

In [0]:
tables = [
    "models",
    "devices",
    "subscription_plans",
    "users",
    "user_updates",
    "conversations",
    "messages",
    "completions",
    "model_inferences",
    "feedback",
    "errors",
    "subscriptions",
    "purchases",
    "payments",
]

final_results = []

for table in tables:
    full_name = f"ai_analytics.bronze.{table}"

    count = spark.table(full_name).count()

    props = {
        row["key"]: row["value"]
        for row in spark.sql(
            f"SHOW TBLPROPERTIES {full_name}"
        ).collect()
    }

    final_results.append(
        (
            table,
            count,
            props.get("delta.appendOnly")
        )
    )

display(
    spark.createDataFrame(
        final_results,
        ["table", "row_count", "append_only"]
    ).orderBy("table")
)